# What is a digital twin good for, and where does each computer fit?

Question: how does a robot work cell go from a scene file, to a twin that follows a real machine, to training data, to a policy that runs at the edge? What does each step measure, and where does it break?

Setup: Kaggle with one Tesla T4 in use (Turing, no bf16, so fp32), internet on, Python 3.13. `usd-core` 26.11 runs in the kernel. `newton` 1.8.0 runs in its own `uv` venv, in a subprocess, with a limit of 8 minutes. If Newton does not install or run, a small NVIDIA Warp 1.19.0 kernel takes its place. PyTorch comes from the Kaggle image. The notebook uses no API key. All random numbers have fixed seeds.

Exam objectives: physical.1 (physical AI versus digital AI, the three-computer pattern, robotics use cases, edge inference), physical.2 (simulation, digital twins, Omniverse, Isaac Sim, synthetic data, domain randomization, the sim-to-real gap, the sim-to-real flywheel), physical.3 (physical AI model families, Cosmos, Isaac GR00T).

Sources for product definitions, all in `hub/sources`: `three-computers-robotics.txt`, `generative-physical-ai.txt`, `physical-ai.txt`, `ai-enterprise.txt`, `humanoid-robots.txt`, `robot-learning.txt`, `synthetic-data-generation.txt`, `closing-the-sim2real-gap-with-nvidia-isaac-sim-and-nvidia-isaac-replicator.txt`, `cosmos.txt`, `gr00t.txt`, `open-models-data-ai.txt`, `difference-between-cloud-and-edge-computing.txt`. The Newton and OpenUSD facts come from their own documentation, read on 2026-11-25.

Hidden truth. The "real" machine in part C is a Python function. Its true numbers sit inside a closure. The twin gets arrays of sensor values and nothing else. The tables print the truth only to show the fit error and to score the alarm. The "real" images in part D come from a second renderer that the training code never calls.

What this notebook does not run:

- NVIDIA Omniverse, NVIDIA Isaac Sim, Cosmos and GR00T. The notebook has no install of them. The OpenUSD files it writes are the data format that Omniverse reads.
- A Jetson. The "edge computer" is one CPU thread that runs a NumPy int8 model. A compiled int8 runtime on real edge hardware is not measured.
- A real robot. The "real" machine is a simulator with hidden numbers.

Terms used below:

- **Physical AI**: AI that perceives, reasons, interacts with and navigates the physical world. **Digital AI** (agentic AI) works in digital environments.
- **Three-computer pattern**: a training computer, a simulation computer, and a computer on the robot.
- **Edge inference**: the trained model runs next to the machine, not in a remote data center.
- **OpenUSD**: a scene format with layers, references, variants and overrides. **UsdPhysics** is its set of physics schemas.
- **Simulation**: a program that computes how the machine moves. **Digital twin**: a simulation of one real machine, kept close to it with sensor data.
- **Residual**: sensor value minus the twin's prediction. An **alarm** fires when the residual statistic crosses a fixed threshold.
- **Synthetic data**: data that a simulator or renderer makes. The label is exact because the scene sets it.
- **Domain randomization**: during training, you draw a new appearance (or physics) for every sample. The model must work on the whole range.
- **Sim-to-real gap**: the loss of accuracy when a model moves from simulated data to real data.
- **Sim-to-real flywheel**: a loop. Log real data, fit the simulator, narrow the randomization, retrain, test again. This is the notebook's working definition. `closing-the-sim2real-gap-with-nvidia-isaac-sim-and-nvidia-isaac-replicator.txt` describes the same idea as iterative tuning of the generated data against real test images.
- **Control period**: the time between two decisions. Here 10 ms. **Delay**: the number of control periods between the sensor reading and the motor command.
- **Wilson interval**: the 95% interval for a rate. **Exact McNemar test**: a paired test for two systems on the same items.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Part | What you measure | Your guess |
|---|---|---|
| A. Scene | Which edits change the source layers? Does a variant change the mass? | |
| B. Physics | Error of the fall distance against `0.5 g t^2`. Speed of 4,096 copies against one copy. | |
| C. Twin | Detection delay and false alarms of the residual alarm, at three thresholds. Twin against a plain speed-per-volt alarm. | |
| C. What-if | Does the twin predict the move time of a worn machine? | |
| D. Synthetic data | Pose error on "real" images, trained without randomization, with it, and after one flywheel turn. | |
| E. Three computers | Control success at the edge, and in the cloud with 20, 50 and 100 ms. Where does it break? | |
| F. Physical AI | Score of a random guess in the matching quiz. | |

In [ ]:
# 1. Setup. Install usd-core before the first import. Constants and helpers that every part uses.
import json, math, os, signal, subprocess, sys, time, traceback
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")          # use one T4

def sh(cmd, timeout, **kw):
    # Run a command with a time limit. Return (exit code, output). A timeout gives code -9 and the output so far.
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)
        return p.returncode, (p.stdout or "") + (p.stderr or "")
    except subprocess.TimeoutExpired as e:
        o = e.stdout or b""
        return -9, (o.decode(errors="replace") if isinstance(o, bytes) else o) + f"\nTIMEOUT after {timeout} s"
    except OSError as e:
        return -1, repr(e)

rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "usd-core==26.11"], 300)
print("pip usd-core==26.11 exit code:", rc, out[-300:] if rc else "")
print(sh(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap,driver_version", "--format=csv"], 30)[1])

import numpy as np, pandas as pd
from scipy.stats import binomtest
from scipy.signal import lfilter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
W = f"{OUT}/twin"; os.makedirs(W, exist_ok=True)
SEED, T0 = 7, time.time()
STATUS, S = [], {}                                          # S keeps results that later parts read

def show(df, name):
    df.to_csv(f"{W}/{name}.csv", index=False)
    print(f"-- {name}"); print(df.to_string(index=False))

def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)

def rate(k, n):
    lo, hi = wilson(int(round(k)), n)
    return f"{int(round(k))}/{n}={100 * k / n:.0f}% [{100 * lo:.0f},{100 * hi:.0f}]"

def mcnemar(a, b):
    # Exact two-sided test on the items where two systems disagree. a, b: boolean arrays on the same items.
    a, b = np.asarray(a, bool), np.asarray(b, bool); x, y = int((a & ~b).sum()), int((~a & b).sum())
    return 1.0 if x + y == 0 else binomtest(x, x + y, 0.5).pvalue

def paired_boot(a, b, n=2000, seed=0):
    # Mean of a - b on the same items, with a 95% bootstrap interval.
    d = np.asarray(a, float) - np.asarray(b, float); r = np.random.default_rng(seed)
    m = [d[r.integers(0, len(d), len(d))].mean() for _ in range(n)]
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

SELF = []
def selfcheck(name, ok):
    # A grader test must pass. A failure is recorded and printed. It does not stop the other parts.
    SELF.append(dict(check=name, ok=bool(ok)))
    if not ok: print("SELF-CHECK FAILED:", name)

# Each grader gets one right and one wrong example.
selfcheck("wilson", wilson(0, 30)[0] == 0 and 0.10 < wilson(0, 30)[1] < 0.12 and wilson(30, 30)[1] == 1.0)
selfcheck("mcnemar", mcnemar([1] * 8, [0] * 8) < 0.01 and mcnemar([1, 0], [0, 1]) == 1.0)
lo, hi = paired_boot(np.arange(50) + 1.0, np.arange(50))[1:]; selfcheck("paired_boot: shifted pair", lo > 0.9 and hi < 1.1)
lo, hi = paired_boot(np.arange(50), np.arange(50))[1:]; selfcheck("paired_boot: equal pair", lo <= 0 <= hi)

class OverBudget(Exception): pass

def part(name, fn, budget_s):
    # Run one part. A failure prints the traceback, is recorded, and the notebook goes on.
    def stop(*a): raise OverBudget(f"over the budget of {budget_s} s")
    signal.signal(signal.SIGALRM, stop); signal.alarm(budget_s)
    t, ok, note = time.time(), True, ""
    try: fn()
    except Exception as e:
        ok, note = False, f"{type(e).__name__}: {str(e)[:140]}"; traceback.print_exc()
    finally: signal.alarm(0)
    STATUS.append(dict(part=name, ok=ok, seconds=round(time.time() - t, 1), budget_s=budget_s, note=note))
    print(f"[{(time.time() - T0) / 60:5.1f} min] part {name}: {'ok' if ok else 'FAILED'}")

## Part A: the work cell as OpenUSD

Question: how does a scene file hold a robot cell, and how do you change it without touching the source files? The exam asks about OpenUSD because it is the data format of NVIDIA Omniverse. Omniverse is described in `ai-enterprise.txt` as libraries and microservices for industrial digital twins and robotics simulation.

The cell has a table, a one-joint arm and a workpiece that falls onto the table.

- **Layers**: `robot_arm.usda` (the asset), `cell.usda` (the cell), `override.usda` (the change). `cell_override.usda` stacks the three.
- **Reference**: the cell pulls the arm in with a reference, so the arm file stays reusable.
- **Variant**: the arm has a `payload` variant set with two masses.
- **Override**: a stronger layer moves the workpiece and picks the heavy payload. Both source files must stay the same, byte for byte.
- **Physics schemas**: `UsdPhysics.Scene`, `RigidBodyAPI`, `CollisionAPI`, `MassAPI` and a `RevoluteJoint`.

The cell below checks five things. The source layers do not change. The override changes the composed position. The variant changes the composed mass. The schema counts are right. The workpiece size that the renderer in part D reads is the size in the file.

In [ ]:
# 2. Part A. Write the layers, stack an override on top, and check what changed.
try:
    from pxr import Gf, Sdf, Usd, UsdGeom, UsdPhysics
except Exception as e:
    print("pxr is not available:", repr(e)[:150])
D = f"{W}/usd"; os.makedirs(D, exist_ok=True)
TABLE_TOP, BOX, DROP, MASS_BOX = 0.40, (0.20, 0.12, 0.10), 0.50, 0.8     # table top height, workpiece size, drop height (m), workpiece mass (kg)
START, MOVED = (0.25, 0.25), (0.15, -0.25)                                # workpiece x, y in the cell and after the override

def unchanged(before, after):
    return all(before[k] == after[k] for k in before)
selfcheck("unchanged: right and wrong", unchanged({"a": "x"}, {"a": "x", "b": "y"}) and not unchanged({"a": "x"}, {"a": "z"}))

def meta(st):
    UsdGeom.SetStageUpAxis(st, UsdGeom.Tokens.z); UsdGeom.SetStageMetersPerUnit(st, 1.0); UsdPhysics.SetStageKilogramsPerUnit(st, 1.0)

def solid(st, path, size, pos, yaw=0.0, mass=None, dynamic=False):
    # A body is an Xform without scale. Its shape is a child Cube that carries the size as a scale.
    body = UsdGeom.Xform.Define(st, path); xf = UsdGeom.XformCommonAPI(body)
    xf.SetTranslate(Gf.Vec3d(*pos)); xf.SetRotate(Gf.Vec3f(0, 0, yaw))
    geom = UsdGeom.Cube.Define(st, path + "/Geom"); geom.GetSizeAttr().Set(1.0)
    UsdGeom.XformCommonAPI(geom).SetScale(Gf.Vec3f(*size))
    UsdPhysics.CollisionAPI.Apply(geom.GetPrim())
    if dynamic: UsdPhysics.RigidBodyAPI.Apply(body.GetPrim())
    if mass: UsdPhysics.MassAPI.Apply(body.GetPrim()).CreateMassAttr(mass)

def make_robot(path):
    st = Usd.Stage.CreateNew(path); meta(st)
    st.SetDefaultPrim(UsdGeom.Xform.Define(st, "/Arm").GetPrim())
    solid(st, "/Arm/Base", (0.16, 0.16, 0.20), (0, 0, 0.10))
    solid(st, "/Arm/Link", (0.50, 0.06, 0.06), (0.25, 0, 0.25), dynamic=True)   # no mass here: the variants set it
    j = UsdPhysics.RevoluteJoint.Define(st, "/Arm/Joint")
    j.CreateBody0Rel().SetTargets([Sdf.Path("/Arm/Base")]); j.CreateBody1Rel().SetTargets([Sdf.Path("/Arm/Link")])
    j.CreateAxisAttr("Z"); j.CreateLocalPos0Attr(Gf.Vec3f(0, 0, 0.15)); j.CreateLocalPos1Attr(Gf.Vec3f(-0.25, 0, 0))
    j.CreateLowerLimitAttr(-170.0); j.CreateUpperLimitAttr(170.0)
    vs = st.GetPrimAtPath("/Arm").GetVariantSets().AddVariantSet("payload")
    for name, kg in (("light", 2.0), ("heavy", 6.0)):
        vs.AddVariant(name); vs.SetVariantSelection(name)
        with vs.GetVariantEditContext():
            UsdPhysics.MassAPI.Apply(st.GetPrimAtPath("/Arm/Link")).CreateMassAttr(kg)
    vs.SetVariantSelection("light"); st.GetRootLayer().Save()

def make_cell(path):
    st = Usd.Stage.CreateNew(path); meta(st)
    st.SetDefaultPrim(UsdGeom.Xform.Define(st, "/World").GetPrim())
    sc = UsdPhysics.Scene.Define(st, "/World/PhysicsScene")
    sc.CreateGravityDirectionAttr(Gf.Vec3f(0, 0, -1)); sc.CreateGravityMagnitudeAttr(9.81)
    solid(st, "/World/Table", (1.0, 0.8, 0.05), (0, 0, TABLE_TOP - 0.025))
    arm = UsdGeom.Xform.Define(st, "/World/Arm"); arm.GetPrim().GetReferences().AddReference("./robot_arm.usda")
    UsdGeom.XformCommonAPI(arm).SetTranslate(Gf.Vec3d(-0.30, 0, TABLE_TOP))
    solid(st, "/World/Workpiece", BOX, (*START, TABLE_TOP + BOX[2] / 2 + DROP), 20.0, MASS_BOX, dynamic=True)
    st.GetRootLayer().Save()

def pos_of(st, path="/World/Workpiece"):
    return tuple(round(float(v), 4) for v in UsdGeom.XformCommonAPI(st.GetPrimAtPath(path)).GetXformVectors(Usd.TimeCode.Default())[0])
def mass_of(st):
    return float(UsdPhysics.MassAPI(st.GetPrimAtPath("/World/Arm/Link")).GetMassAttr().Get())

def partA():
    make_robot(f"{D}/robot_arm.usda"); make_cell(f"{D}/cell.usda")
    src = Usd.Stage.Open(f"{D}/cell.usda")                               # the cell alone, without the override
    ov = Sdf.Layer.CreateNew(f"{D}/override.usda"); root = Sdf.Layer.CreateNew(f"{D}/cell_override.usda")
    root.subLayerPaths.append("./override.usda"); root.subLayerPaths.append("./cell.usda"); root.defaultPrim = "World"
    st = Usd.Stage.Open(root); meta(st)
    snap = lambda: {l.identifier.rsplit("/", 1)[-1]: l.ExportToString() for l in st.GetUsedLayers()}
    before = snap()
    st.SetEditTarget(Usd.EditTarget(ov))                                  # every edit from here goes into override.usda
    UsdGeom.XformCommonAPI(st.GetPrimAtPath("/World/Workpiece")).SetTranslate(Gf.Vec3d(*MOVED, TABLE_TOP + BOX[2] / 2 + DROP))
    st.GetPrimAtPath("/World/Arm").GetVariantSet("payload").SetVariantSelection("heavy")
    ov.Save(); root.Save(); after = snap()
    keep = ("cell.usda", "robot_arm.usda")
    assert unchanged({k: before[k] for k in keep}, {k: after[k] for k in keep}), "the override changed a source layer"
    assert before["override.usda"] != after["override.usda"]
    for n in ("robot_arm", "cell", "override", "cell_override"):
        print(f"===== {n}.usda\n{open(f'{D}/{n}.usda').read()}")
    rows = []
    for p in st.Traverse():
        apis = [a[:-3] for a in ("RigidBodyAPI", "CollisionAPI", "MassAPI") if p.HasAPI(getattr(UsdPhysics, a))]
        rows.append(dict(path=str(p.GetPath()), type=p.GetTypeName(), schemas=",".join(apis)))
    show(pd.DataFrame(rows), "a_prims")
    n_body = sum("RigidBody" in r["schemas"] for r in rows); n_col = sum("Collision" in r["schemas"] for r in rows)
    n_joint = sum(r["type"] == "PhysicsRevoluteJoint" for r in rows)
    box = tuple(float(v) for v in UsdGeom.XformCommonAPI(st.GetPrimAtPath("/World/Workpiece/Geom")).GetXformVectors(Usd.TimeCode.Default())[2])
    chk = [("source layers unchanged after the override", True),
           ("workpiece x,y: cell -> override", f"{pos_of(src)[:2]} -> {pos_of(st)[:2]}", pos_of(src)[:2] == START and pos_of(st)[:2] == MOVED),
           ("arm payload mass (kg): light -> heavy", f"{mass_of(src)} -> {mass_of(st)}", (mass_of(src), mass_of(st)) == (2.0, 6.0)),
           ("rigid bodies, colliders, joints", f"{n_body}, {n_col}, {n_joint}", (n_body, n_col, n_joint) == (2, 4, 1)),
           ("workpiece size read from the scene (m)", str(box), bool(np.allclose(box, BOX, atol=1e-5)))]
    df = pd.DataFrame([dict(check=c[0], value=c[1] if len(c) == 3 else "", passed=c[-1]) for c in chk]); show(df, "a_checks")
    assert df.passed.all(), "a scene check failed"
    S["scene"] = f"{D}/cell_override.usda"; S["box"] = box
part("A scene", partA, 120)

## Part B: physics, and many copies at once

Question: can a physics engine load the scene, and does it agree with a formula? How much faster are many copies than one? The exam asks about simulation because a simulation computer must run many machines in parallel.

`physical-ai.txt` lists Newton as a GPU-accelerated physics engine. The Newton documentation (`newton-physics.github.io`, read on 2026-11-25) says that it builds on NVIDIA Warp, and that `ModelBuilder.add_usd` loads OpenUSD. USD import needs `usd-core` and `newton-usd-schemas`. Its own extras pin `usd-core` below 26.9, so this try installs `newton`, `usd-core` 26.11 and `newton-usd-schemas` by hand. If that fails, the printed error is the result.

The try is bounded to 8 minutes in total for the install and the run. It runs in a `uv` venv and a subprocess, so the kernel never imports Newton or its Warp. If the USD import fails, Newton builds an equal scene with its Python API. If Newton does not work at all, a 20-line Warp kernel runs the same drop test.

Two checks use formulas. The fall distance at 0.25 s must be `0.5 g t^2`. The workpiece must come to rest on the table, with its centre at table top plus half its height. The start height must equal the height in the scene file. Speed is world-steps per second for 1 to 4,096 copies. One world-step is one copy advanced by one step.

In [ ]:
# 3. Part B (definitions). The Newton script, the Warp fallback script, and the checks.
NEWTON_PY = r'''
import json, sys, time, traceback
from importlib.metadata import version
import warp as wp
import newton
from newton.solvers import SolverXPBD
cfg = json.loads(sys.argv[1]); DT = cfg["dt"]
def emit(**kw): print("RESULT " + json.dumps(kw), flush=True)
emit(kind="versions", engine="newton", newton=version("newton"), warp=wp.__version__, device=str(wp.get_device()))

def cell(use_usd):
    b = newton.ModelBuilder()
    if use_usd:
        body = b.add_usd(cfg["scene"])["path_body_map"]["/World/Workpiece"]
    else:
        top, (bx, by, bz) = cfg["top"], cfg["box"]
        b.add_shape_box(-1, xform=wp.transform(wp.vec3(0.0, 0.0, top - 0.025), wp.quat_identity()), hx=0.5, hy=0.4, hz=0.025)
        body = b.add_body(xform=wp.transform(wp.vec3(*cfg["moved"], top + bz / 2 + cfg["drop"]), wp.quat_identity()), mass=cfg["mass"])
        b.add_shape_box(body, hx=bx / 2, hy=by / 2, hz=bz / 2)
    return b, body

def run(b, body, copies, steps, probes):
    big = newton.ModelBuilder(); big.replicate(b, copies)
    model = big.finalize()
    solver = SolverXPBD(model, iterations=cfg["iters"])
    s0, s1, ctrl = model.state(), model.state(), model.control()
    newton.eval_fk(model, model.joint_q, model.joint_qd, s0)
    pipe = newton.CollisionPipeline(model); contacts = pipe.contacts()
    z0, z = float(s0.body_q.numpy()[body, 2]), {}
    wp.synchronize(); t0 = time.perf_counter()
    for k in range(1, steps + 1):
        s0.clear_forces(); pipe.collide(s0, contacts); solver.step(s0, s1, ctrl, contacts, DT); s0, s1 = s1, s0
        if k in probes: z[k] = float(s0.body_q.numpy()[body, 2])
    wp.synchronize()
    return time.perf_counter() - t0, z0, z, model

def go(use_usd):
    src = "usd" if use_usd else "api"
    b, body = cell(use_usd)
    run(b, body, 1, 3, [])                                   # warm-up: compiles the kernels
    sec, z0, z, model = run(b, body, 1, cfg["steps"], cfg["probes"])
    emit(kind="check", engine="newton", source=src, z0=z0, z=z, seconds=sec, bodies=getattr(model, "body_count", None),
         shapes=getattr(model, "shape_count", None), joints=getattr(model, "joint_count", None))
    for n in cfg["copies"]:
        run(b, body, n, 3, [])                                   # warm-up at this size: the first run of a size compiles and allocates
        emit(kind="speed", engine="newton", source=src, copies=n, steps=cfg["speed_steps"], seconds=run(b, body, n, cfg["speed_steps"], [])[0])

for use_usd in (True, False):
    try:
        go(use_usd); break
    except Exception:
        emit(kind="error", engine="newton", source="usd" if use_usd else "api", error=traceback.format_exc()[-700:])
'''
WARP_PY = r'''
import json, sys, time
import warp as wp
wp.config.quiet = True
cfg = json.loads(sys.argv[1]); dt, h, top = cfg["dt"], cfg["box"][2], cfg["top"]
z0, rest = top + h / 2 + cfg["drop"], top + h / 2        # centre height at the start and at rest

@wp.kernel
def drop(z0: float, rest: float, dt: float, n: int, probe: int, zp: wp.array[float], ze: wp.array[float]):
    i = wp.tid()
    z = float(z0)
    v = float(0.0)
    for s in range(n):
        a = float(-9.81)
        if z < rest:
            a = a + 40000.0 * (rest - z) - 400.0 * v      # a spring and a damper hold the box on the table
        v = v + a * dt
        z = z + v * dt
        if s + 1 == probe:
            zp[i] = z
    ze[i] = z

def run(copies, steps, probe):
    zp, ze = wp.zeros(copies, dtype=float), wp.zeros(copies, dtype=float)
    wp.synchronize(); t0 = time.perf_counter()
    wp.launch(drop, dim=copies, inputs=[z0, rest, dt, steps, probe, zp, ze])
    wp.synchronize()
    return time.perf_counter() - t0, float(zp.numpy()[0]), float(ze.numpy()[0])

def emit(**kw): print("RESULT " + json.dumps(kw), flush=True)
emit(kind="versions", engine="warp", warp=wp.__version__, device=str(wp.get_device()))
run(1, 3, 2)                                                  # warm-up: compiles the kernel
sec, zp, ze = run(1, cfg["steps"], cfg["probes"][0])
emit(kind="check", engine="warp", source="kernel", z0=z0, z={str(cfg["probes"][0]): zp, str(cfg["steps"]): ze}, seconds=sec)
for n in cfg["copies"]:
    run(n, 3, 1)                                             # warm-up at this size
    emit(kind="speed", engine="warp", source="kernel", copies=n, steps=cfg["steps"], seconds=run(n, cfg["steps"], 1)[0])
'''

def within(measured, expected, tol, relative):
    # One grader for both checks: relative error (fall distance) or absolute error in metres (heights).
    err = abs(measured - expected) / abs(expected) if relative else abs(measured - expected)
    return err, err <= tol
selfcheck("within: right and wrong relative", within(1.0, 1.0, 0.02, True)[1] and not within(1.1, 1.0, 0.02, True)[1])
selfcheck("within: right and wrong absolute", within(0.45, 0.452, 0.01, False)[1] and not within(0.5, 0.45, 0.01, False)[1])

def check_rows(r, c):
    # Three checks from one result row: start height, fall distance at t1, rest height.
    g, t1, nfree, nend = 9.81, c["probes"][0] * c["dt"], str(c["probes"][0]), str(c["steps"])
    exp_z0 = c["top"] + c["box"][2] / 2 + c["drop"]
    fall, rest = r["z0"] - r["z"][nfree], c["top"] + c["box"][2] / 2
    out = []
    for name, formula, exp, got, tol, rel in (
            ("start height (m)", "scene file", exp_z0, r["z0"], 1e-3, False),
            (f"fall distance at {t1:.2f} s (m)", "0.5 g t^2", 0.5 * g * t1 ** 2, fall, 0.02, True),
            ("rest height of the centre (m)", "table top + h/2", rest, r["z"][nend], 0.01, False)):
        err, ok = within(got, exp, tol, rel)
        out.append(dict(engine=r["engine"], source=r["source"], quantity=name, formula=formula, expected=round(exp, 4),
                        measured=round(got, 4), error=round(err, 4), tolerance=tol, passed=ok))
    return out

In [ ]:
# 4. Part B (run). Try Newton for at most 8 minutes. If it gives no check row, run the Warp kernel instead.
CFG_B = dict(scene=S.get("scene", f"{D}/cell_override.usda"), dt=0.001, iters=8, steps=1500, probes=[250, 1500], speed_steps=200,
             copies=[1, 64, 1024, 4096], top=TABLE_TOP, box=list(BOX), drop=DROP, mass=MASS_BOX, moved=list(MOVED))

def rows_of(out): return [json.loads(l[7:]) for l in out.splitlines() if l.startswith("RESULT ")]

def engine_run(name, cmd, limit):
    rc, out = sh(cmd, limit, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    rows = rows_of(out)
    print(f"{name}: exit code {rc}, {len(rows)} result rows")
    if rc != 0 or not any(r["kind"] == "check" for r in rows): print(out[-1800:])         # the real error
    return rows

def partB():
    end, rows = time.time() + 480, []
    left = lambda: max(20, int(end - time.time()))
    venv = f"{OUT}/newton_venv"; script = f"{W}/newton_run.py"
    rc, out = sh(["uv", "venv", venv], 120); print("uv venv exit code:", rc, out[-200:] if rc else "")
    if rc == 0:
        rc, out = sh(["uv", "pip", "install", "--python", f"{venv}/bin/python", "newton==1.8.0", "usd-core==26.11", "newton-usd-schemas"], left())
        print("install newton exit code:", rc, out[-1200:] if rc else "")
    if rc == 0:
        open(script, "w").write(NEWTON_PY)
        rows = engine_run("newton", [f"{venv}/bin/python", script, json.dumps(CFG_B)], left())
    if not any(r["kind"] == "check" for r in rows):
        print("Newton gave no result. The Warp kernel runs the same drop test.")
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "warp-lang==1.19.0"], 300)
        print("install warp-lang exit code:", rc, out[-300:] if rc else "")
        open(f"{W}/warp_drop.py", "w").write(WARP_PY)
        if rc == 0: rows = engine_run("warp", [sys.executable, f"{W}/warp_drop.py", json.dumps(CFG_B)], 300)
    print("versions:", [r for r in rows if r["kind"] == "versions"])
    for r in rows:
        if r["kind"] == "error": print(f"newton error with source={r['source']}:\n{r['error']}")
    pd.DataFrame(rows).astype(str).to_csv(f"{W}/b_raw_rows.csv", index=False)
    chk = [x for r in rows if r["kind"] == "check" for x in check_rows(r, CFG_B)]
    assert chk, "no engine produced a check row"
    show(pd.DataFrame(chk), "b_checks")
    sp = pd.DataFrame([r for r in rows if r["kind"] == "speed"])
    sp["world_steps_per_s"] = (sp.copies * sp.steps / sp.seconds).round(0)
    one = sp.groupby(["engine", "source"]).world_steps_per_s.transform("first")
    sp["x_vs_one_copy"] = (sp.world_steps_per_s / one).round(1); sp["seconds"] = sp.seconds.round(3)
    show(sp[["engine", "source", "copies", "steps", "seconds", "world_steps_per_s", "x_vs_one_copy"]], "b_speed")
    assert all(c["passed"] for c in chk), "a physics check failed"
part("B physics", partB, 840)

## Part C: a twin that follows a real machine

Question: can a twin that sees only the sensor stream estimate the machine, follow it, and raise an alarm when the machine wears? `generative-physical-ai.txt` says digital twins represent real machines and spaces, so teams can test how robots and sensors interact.

The "real" machine is the arm joint. Its speed follows `d(speed) = dt * (alpha * volts - beta * speed - gamma * tanh(speed / 0.05))`. Here `alpha` is motor gain per inertia, `beta` is viscous friction, and `gamma` is Coulomb friction. Each machine has its own hidden numbers, within 5% of 4.0, 2.5 and 0.8. The sensors are the angle (noise 0.002 rad) and the speed (noise 0.02 rad/s) at 100 Hz. The command is a random piecewise-constant voltage.

- **Estimate**: least squares on the first 60 s, which are healthy.
- **Residual alarm**: the twin predicts the next speed from the measured speed and the command. The residual is the difference. The statistic is a moving average (1 s) of the squared residual, divided by its healthy value. It is 1 when nothing is wrong. The alarm fires when it crosses a fixed threshold.
- **Wear**: from 100 s the friction `beta` rises to 3 times its value over 40 s. The twin never reads this.
- **Setting on one half, measurement on the other**: the threshold is the smallest one in the grid with zero alarms on 30 calibration machines (seeds 100 to 129, no wear). It is then measured on 30 other machines (seeds 200 to 229), once without wear (false alarms) and once with wear (detection).
- **Baseline without a twin**: speed per volt. It is a moving average (2 s) of the absolute speed divided by a moving average of the absolute command. The threshold is chosen the same way. Two trivial answers are printed next to it: always silent and always alarm.
- **What-if**: the twin answers "what if the friction doubles, and which voltage still moves the arm 1 rad in 0.45 s?". The real machine then runs both voltages.

In [ ]:
# 5. Part C (definitions). The hidden machine, the twin (pure functions of arrays), the alarm, and the graders.
DT, EPS, U_MAX = 0.01, 0.05, 5.0                                      # control period (s), friction smoothing, voltage limit (V)
RUN_S, CAL_S, WEAR_AT, WEAR_X, RAMP = 180, 60, 100, 3.0, 40           # seconds; wear multiplies beta by 3 over 40 s
N_RUN, N_CAL = int(RUN_S / DT), int(CAL_S / DT)
CAL_SEEDS, TEST_SEEDS = range(100, 130), range(200, 230)
THR_TWIN, THR_RAW = [1.5, 2.0, 3.0], [1.1, 1.2, 1.3, 1.5, 2.0]

def make_machine(seed, wear_at=None, wear_x=WEAR_X, ramp=RAMP):
    # The real machine. Its true numbers live in this closure. Only the scoring code reads `truth`.
    rng = np.random.default_rng(seed)
    a, b, g = (v * (1 + 0.05 * rng.standard_normal()) for v in (4.0, 2.5, 0.8))
    s = dict(th=0.0, w=0.0, t=0.0)
    beta = lambda t: b if wear_at is None else b * (1 + (wear_x - 1) * min(1.0, max(0.0, (t - wear_at) / ramp)))
    def step(u):
        u = min(max(float(u), -U_MAX), U_MAX)
        s["w"] += DT * (a * u - beta(s["t"]) * s["w"] - g * math.tanh(s["w"] / EPS)); s["th"] += DT * s["w"]; s["t"] += DT
        return s["th"] + 0.002 * rng.standard_normal(), s["w"] + 0.02 * rng.standard_normal()
    return step, dict(alpha=a, beta=b, gamma=g, beta_at=beta)

def commands(seed, n):
    r = np.random.default_rng(10_000 + seed); u = np.empty(n); i = 0
    while i < n:
        h = int(r.integers(30, 100)); u[i:i + h] = r.uniform(-3, 3); i += h
    return u

def record(seed, wear_at=None, n=N_RUN):
    step, truth = make_machine(seed, wear_at); u = commands(seed, n)
    return u, np.array([step(x) for x in u]), truth                   # command, (angle, speed) sensors, truth for scoring

# The twin. These functions get arrays and parameters. They cannot reach the machine.
def feats(u, w): return np.column_stack([u, w, np.tanh(w / EPS)])
def fit_params(u, w):
    c = np.linalg.lstsq(feats(u[1:], w[:-1]), np.diff(w), rcond=None)[0] / DT      # u[t+1] acts between the readings w[t] and w[t+1]
    return np.array([c[0], -c[1], -c[2]])                             # alpha, beta, gamma
def residual(u, w, th):
    return np.diff(w) - DT * feats(u[1:], w[:-1]) @ np.array([th[0], -th[1], -th[2]])
def ewma(x, lam, start=1.0): return lfilter([lam], [1, lam - 1], x, zi=[(1 - lam) * start])[0]
def twin_stat(u, y):
    th = fit_params(u[:N_CAL], y[:N_CAL, 1]); r = residual(u, y[:, 1], th)
    return ewma((r / r[:N_CAL - 1].std()) ** 2, 0.01), th
def raw_stat(u, y):
    ratio = ewma(abs(y[:, 1]), 0.005, abs(y[:N_CAL, 1]).mean()) / ewma(abs(u), 0.005, abs(u[:N_CAL]).mean())
    return np.median(ratio[N_CAL // 2:N_CAL]) / ratio            # about 1 when healthy, above 1 when the arm is slower per volt

def first_alarm(e, thr, start=N_CAL):
    i = np.flatnonzero(e[start:] > thr); return int(i[0]) + start if len(i) else -1
e_t = np.ones(1000); e_t[600:] = 3.0
selfcheck("first_alarm", first_alarm(e_t, 2.0, 100) == 600 and first_alarm(e_t, 5.0, 100) == -1 and first_alarm(e_t, 2.0, 700) == 700)

def sim_move(theta, u, target=1.0, t_max=1.5):
    # The twin's own prediction of one move: seconds to reach the target angle with a constant voltage.
    a, b, g = theta; th = w = 0.0
    for k in range(int(t_max / DT)):
        w += DT * (a * u - b * w - g * math.tanh(w / EPS)); th += DT * w
        if th >= target: return (k + 1) * DT
    return math.inf
def real_move(seed, wear_x, u, target=1.0, t_max=1.5):
    step, _ = make_machine(seed, 0.0, wear_x, 1e-9)                   # wear from the first tick, at full size
    for k in range(int(t_max / DT)):
        if step(u)[0] >= target: return (k + 1) * DT
    return math.inf
selfcheck("sim_move: fast and too weak", sim_move((4.0, 2.5, 0.8), 4.0) < 0.45 and sim_move((4.0, 2.5, 0.8), 0.1) == math.inf)

In [ ]:
# 6. Part C (run). Assert the simulator, estimate, follow, alarm, and answer the what-if.
def partC():
    # (a) Assert the truth before use: the steady speed of a machine equals the root of alpha*u = beta*w + gamma*tanh(w/EPS).
    from scipy.optimize import brentq
    step, tr = make_machine(1)
    for _ in range(1500): step(2.0)
    w_ss = np.mean([step(2.0)[1] for _ in range(200)])
    w_f = brentq(lambda w: tr["alpha"] * 2.0 - tr["beta"] * w - tr["gamma"] * math.tanh(w / EPS), 0.0, 20.0)
    assert abs(w_ss - w_f) / w_f < 0.01, f"steady speed {w_ss:.3f} against formula {w_f:.3f}"
    wearing = make_machine(1, WEAR_AT)[1]["beta_at"]
    assert wearing(WEAR_AT) == tr["beta"] and abs(wearing(WEAR_AT + RAMP) - WEAR_X * tr["beta"]) < 1e-9
    print(f"steady speed at 2 V: simulator {w_ss:.3f} rad/s, formula {w_f:.3f} rad/s. Wear schedule checked.")
    # (b) Estimate: relative error of the fitted numbers on the 30 test machines (healthy first minute).
    runs = {s: record(s, WEAR_AT) for s in TEST_SEEDS}; quiet = {s: record(s) for s in TEST_SEEDS}
    err = []
    for s in TEST_SEEDS:
        u, y, tr = runs[s]; th = fit_params(u[:N_CAL], y[:N_CAL, 1])
        err.append([abs(th[i] - tr[k]) / tr[k] for i, k in enumerate(("alpha", "beta", "gamma"))])
    err = np.array(err); S["theta"] = fit_params(runs[200][0][:N_CAL], runs[200][1][:N_CAL, 1])
    show(pd.DataFrame(dict(number=["alpha", "beta", "gamma"], median_rel_error=np.median(err, 0).round(4), worst_rel_error=err.max(0).round(4))), "c_estimate")
    # (c) Follow: refit on the last 20 s of one wear run. The true beta is read only here, to print the error.
    u, y, tr = runs[200]; rows = []
    for t in (60, 80, 100, 120, 140, 160, 180):
        i = int(t / DT); th = fit_params(u[i - 2000:i], y[i - 2000:i, 1]); true = tr["beta_at"](t - 10)
        rows.append(dict(t_s=t, twin_beta=round(th[1], 3), true_beta_mid_window=round(true, 3), rel_error=round(abs(th[1] - true) / true, 3)))
    show(pd.DataFrame(rows), "c_follow")
    # (d) Alarm: pick each threshold on the calibration machines, then measure on the test machines.
    stat = lambda kind, ur: twin_stat(*ur[:2])[0] if kind == "twin" else raw_stat(*ur[:2])
    res, paired, dl, plot, onset = [], {}, {}, {}, int(WEAR_AT / DT)
    for kind, grid in (("twin", THR_TWIN), ("raw", THR_RAW)):
        cal = [stat(kind, record(s)) for s in CAL_SEEDS]
        chosen = next((t for t in grid if all(first_alarm(e, t) < 0 for e in cal)), grid[-1])
        quiet_e = {s: stat(kind, quiet[s]) for s in TEST_SEEDS}; wear_e = {s: stat(kind, runs[s]) for s in TEST_SEEDS}
        plot[kind] = (quiet_e[200], wear_e[200], chosen)
        for thr in grid:
            fa = sum(first_alarm(e, thr) >= 0 for e in quiet_e.values())                        # alarms on machines without wear
            early = sum(0 <= first_alarm(e, thr) < onset for e in wear_e.values())              # alarms before the wear starts
            hit = {s: first_alarm(e, thr, onset) for s, e in wear_e.items()}
            det = [s for s, i in hit.items() if i >= 0]
            delay = [hit[s] * DT - WEAR_AT for s in det]
            mult = [runs[s][2]["beta_at"](hit[s] * DT) / runs[s][2]["beta"] for s in det]       # scoring only: how worn was it at the alarm
            res.append(dict(method=kind, threshold=thr, chosen=thr == chosen, false_alarm_runs=rate(fa, 30), detected=rate(len(det), 30),
                            early_alarms=early, median_delay_s=round(float(np.median(delay)), 1) if delay else None,
                            wear_at_alarm=round(float(np.median(mult)), 2) if mult else None))
            if thr == chosen: paired[kind] = [hit[s] >= 0 for s in TEST_SEEDS]; dl[kind] = {s: hit[s] * DT - WEAR_AT for s in det}
    res += [dict(method="always silent", false_alarm_runs=rate(0, 30), detected=rate(0, 30)),
            dict(method="always alarm", false_alarm_runs=rate(30, 30), detected=rate(30, 30))]
    show(pd.DataFrame(res), "c_alarm")
    print(f"detections at the chosen thresholds, twin against raw baseline: exact McNemar p = {mcnemar(paired['twin'], paired['raw']):.3f}")
    both = sorted(set(dl["twin"]) & set(dl["raw"])); d, lo, hi = paired_boot([dl["raw"][s] for s in both], [dl["twin"][s] for s in both])
    print(f"detection delay, raw minus twin, on {len(both)} machines that both detect: mean {d:.1f} s, 95% bootstrap interval [{lo:.1f}, {hi:.1f}]")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    for a_, k in zip(ax, ("twin", "raw")):
        q, w_, c = plot[k]; tt = np.arange(len(q)) * DT
        a_.plot(tt, q, label="no wear"); a_.plot(tt, w_, label="wear from 100 s"); a_.axhline(c, color="k", ls="--", label="chosen threshold")
        a_.set_title(f"{k} statistic"); a_.set_xlabel("s"); a_.set_ylim(0, 6); a_.legend()
    fig.tight_layout(); fig.savefig(f"{W}/c_alarm.png", dpi=110); plt.close(fig)
    # (e) What-if on the twin, then the check on the real machine (seed 200).
    th, goal = S["theta"], 0.45
    volts = lambda t: next((v for v in np.arange(0.5, 5.01, 0.1) if sim_move(t, v) <= goal), math.nan)
    th2 = th * np.array([1.0, 2.0, 1.0]); u_h, u_w = volts(th), volts(th2)
    df = pd.DataFrame([dict(case="healthy voltage, healthy machine", volts=u_h, twin_s=sim_move(th, u_h), real_s=real_move(200, 1.0, u_h)),
                       dict(case="healthy voltage, friction doubled", volts=u_h, twin_s=sim_move(th2, u_h), real_s=real_move(200, 2.0, u_h)),
                       dict(case="twin's voltage, friction doubled", volts=u_w, twin_s=sim_move(th2, u_w), real_s=real_move(200, 2.0, u_w))])
    df["goal_s"] = goal; df["twin_error_s"] = (df.twin_s - df.real_s).abs(); df["real_meets_goal"] = df.real_s <= goal
    show(df.round(3), "c_whatif")
part("C twin", partC, 240)

## Part D: synthetic data, domain randomization, and the sim-to-real gap

Question: a vision model must find the pose of the workpiece (centre and angle) from a top-down camera. You have no labelled real images. Can synthetic images do the job? `synthetic-data-generation.txt` names synthetic data from simulations as a way to train when real data is scarce. `closing-the-sim2real-gap-with-nvidia-isaac-sim-and-nvidia-isaac-replicator.txt` says domain randomization widens the generated data so that it covers the real range. It splits the gap into an appearance gap and a content gap. The next cells test the appearance gap.

- **Scene to image**: the renderer reads the table size and the workpiece size from the OpenUSD stage of part A. The label (x, y, angle) is exact, because the scene sets it. The code asserts this before it trains.
- **Simulated renderer**: soft edges, an optional random look per image. **"Real" renderer**: a second function with hard edges, a drop shadow, a textured table, cables, vignetting, noise and 4-bit levels. The training code never calls it.
- **Three training sets of 6,000 images**: (1) fixed look, no randomization. (2) domain randomization: random colours, light gradient, blur, noise and distractors. (3) one flywheel turn: 40 labelled "real" images give estimates of the table colour, the workpiece colour and the noise. The randomization then narrows around the estimates.
- **Model**: a small CNN on the T4. It outputs x, y, `cos 2a` and `sin 2a`, because a rectangle looks the same after 180 degrees. Three seeds each.
- **Tests**: 400 clean simulated images, and 400 "real" images that are not the 40 above. Success means position error below 2 cm and angle error below 10 degrees. The baseline always answers the mean training pose.

In [ ]:
# 7. Part D (definitions). Two renderers, the pose metric, the CNN, and the flywheel estimate.
from scipy.ndimage import gaussian_filter, uniform_filter
VIEW, NPX, HALF = 0.8, 48, 0.25                                       # camera view (m), image size (px), pose range (m)
GX, GY = np.meshgrid((np.arange(NPX) + 0.5) / NPX * VIEW - VIEW / 2, (np.arange(NPX) + 0.5) / NPX * VIEW - VIEW / 2)

def coverage(pose, size, soft):
    # Share of each pixel that a rectangle covers. pose: (n,3) = x, y, angle. size: (2,) or (n,2) in metres. soft: edge width in pixels, 0 = hard.
    n, size = len(pose), np.broadcast_to(np.asarray(size, float), (len(pose), 2))
    dx, dy = GX[None] - pose[:, 0, None, None], GY[None] - pose[:, 1, None, None]
    c, s = np.cos(pose[:, 2])[:, None, None], np.sin(pose[:, 2])[:, None, None]
    u, v = abs(c * dx + s * dy), abs(-s * dx + c * dy)
    hx, hy = size[:, 0, None, None] / 2, size[:, 1, None, None] / 2
    if soft == 0: return ((u < hx) & (v < hy)).astype(np.float32)
    k = VIEW / NPX * soft
    return (np.clip((hx - u) / k + .5, 0, 1) * np.clip((hy - v) / k + .5, 0, 1)).astype(np.float32)

def sample_pose(n, rng): return np.column_stack([rng.uniform(-HALF, HALF, n), rng.uniform(-HALF, HALF, n), rng.uniform(0, np.pi, n)])
def encode(p): return np.column_stack([p[:, 0] / HALF, p[:, 1] / HALF, np.cos(2 * p[:, 2]), np.sin(2 * p[:, 2])]).astype(np.float32)
def decode(P): return np.column_stack([P[:, 0] * HALF, P[:, 1] * HALF, np.arctan2(P[:, 3], P[:, 2]) / 2])
def pose_err(P, true):
    # Position error in cm and angle error in degrees. A rectangle repeats every 180 degrees, so the angle error is at most 90.
    d = decode(P); pos = 100 * np.hypot(d[:, 0] - true[:, 0], d[:, 1] - true[:, 1])
    return pos, np.degrees(np.abs(np.angle(np.exp(2j * (d[:, 2] - true[:, 2]))))) / 2
def success(P, true): pos, ang = pose_err(P, true); return (pos < 2.0) & (ang < 10.0)
_p = sample_pose(5, np.random.default_rng(0))
selfcheck("pose_err: exact pose", pose_err(encode(_p), _p)[0].max() < 1e-4 and pose_err(encode(_p), _p)[1].max() < 0.01 and success(encode(_p), _p).all())
_q = _p + np.array([0.0, 0.0, np.pi / 2]); selfcheck("pose_err: angle off by 90 degrees", np.allclose(pose_err(encode(_q), _p)[1], 90, atol=0.01) and not success(encode(_q), _p).any())
_q = _p + np.array([0.1, 0.0, 0.0]); selfcheck("pose_err: position off by 10 cm", np.allclose(pose_err(encode(_q), _p)[0], 10, atol=0.01))

def sample_app(kind, n, rng, est=None):
    # The look of the simulated renderer. fixed: one look. dr: a random look per image. fit: random, but near an estimate from real images.
    if kind == "fixed":
        return dict(bg=np.tile([.55, .55, .55], (n, 1)), obj=np.tile([.85, .35, .15], (n, 1)), noise=np.full(n, .01),
                    grad=np.zeros((n, 2)), blur=np.zeros(n), nd=np.zeros(n, int))
    if kind == "dr":
        bg = rng.uniform(.2, .9, (n, 3)); sign = np.where(bg.mean(1, keepdims=True) > .55, -1, 1)
        obj = np.clip(bg + sign * rng.uniform(.25, .5, (n, 3)), 0, 1); noise = rng.uniform(0, .08, n)
    else:
        bg = np.clip(est["bg"] + rng.uniform(-.1, .1, (n, 3)), 0, 1); obj = np.clip(est["obj"] + rng.uniform(-.1, .1, (n, 3)), 0, 1)
        noise = est["noise"] * rng.uniform(.5, 1.5, n)
    return dict(bg=bg, obj=obj, noise=noise, grad=rng.uniform(-.3, .3, (n, 2)), blur=rng.choice([0., .7, 1.4], n), nd=rng.integers(0, 4, n))

def render_sim(pose, app, rng, box):
    n = len(pose); img = np.broadcast_to(app["bg"][:, None, None, :], (n, NPX, NPX, 3)).copy()
    for d in range(3):                                                          # up to 3 small distractors, drawn under the workpiece
        dp = sample_pose(n, rng); m = coverage(dp, rng.uniform(.03, .08, (n, 2)), 1.0) * (app["nd"] > d)[:, None, None]
        img = img * (1 - m[..., None]) + rng.uniform(0, 1, (n, 1, 1, 3)) * m[..., None]
    m = coverage(pose, box[:2], 1.0)[..., None]
    img = img * (1 - m) + app["obj"][:, None, None, :] * m
    img *= 1 + 2.5 * (app["grad"][:, 0, None, None, None] * GX[None, ..., None] + app["grad"][:, 1, None, None, None] * GY[None, ..., None])
    for s in (.7, 1.4):
        i = app["blur"] == s
        if i.any(): img[i] = gaussian_filter(img[i], (0, s, s, 0))
    img = img + app["noise"][:, None, None, None] * rng.standard_normal(img.shape)
    return np.clip(img, 0, 1).transpose(0, 3, 1, 2).astype(np.float32)

def render_real(pose, rng, box):
    # A different renderer: hard edges, a drop shadow, a textured table, two cables, vignetting, noise, 4-bit levels.
    n = len(pose); m = coverage(pose, box[:2], 0)[..., None]; sh = coverage(pose + np.array([.03, -.03, 0]), box[:2], 0)[..., None]
    img = np.array([.36, .43, .50]) + 0.5 * gaussian_filter(rng.standard_normal((n, NPX, NPX)), (0, 4, 4))[..., None]
    img = img * (1 - .35 * sh)
    for _ in range(2):
        cp = sample_pose(n, rng); cm = coverage(cp, (.20, .012), 0)[..., None]; img = img * (1 - cm) + np.array([.2, .15, .1]) * cm
    img = img * (1 - m) + np.array([.72, .52, .31]) * m
    img *= (1 - .3 * (GX ** 2 + GY ** 2) / (VIEW / 2) ** 2)[None, ..., None]
    img = img + .04 * rng.standard_normal(img.shape)
    return (np.round(np.clip(img, 0, 1) * 15) / 15).transpose(0, 3, 1, 2).astype(np.float32)

def fit_look(img, pose, box):
    # Flywheel step: estimate the table colour, workpiece colour and noise from labelled real images.
    m = coverage(pose, box[:2], 0)[:, None]
    bg = (img * (1 - m)).sum((0, 2, 3)) / (1 - m).sum((0, 2, 3)); obj = (img * m).sum((0, 2, 3)) / m.sum((0, 2, 3))
    return dict(bg=bg, obj=obj, noise=float((img - uniform_filter(img, (1, 1, 3, 3))).std()))

def fit_cnn(Xtr, ytr, tests, seed, epochs=150):   # 30 epochs left the domain-randomization network undertrained (first run: 13 cm error on clean simulated images)
    # A small CNN on the GPU. tests: dict of name -> images. Returns the predictions for every test set.
    import torch, torch.nn as nn
    dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(seed)
    blk = lambda i, o: [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2)]
    net = nn.Sequential(*blk(3, 16), *blk(16, 32), *blk(32, 64), nn.Flatten(), nn.Linear(64 * 36, 128), nn.ReLU(), nn.Linear(128, 4)).to(dev)
    X, Y = torch.tensor(Xtr, device=dev), torch.tensor(ytr, device=dev)
    opt = torch.optim.Adam(net.parameters(), 2e-3); steps = epochs * ((len(X) + 127) // 128)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 4e-3, total_steps=steps); g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        net.train(); perm = torch.randperm(len(X), generator=g).to(dev)
        for i in range(0, len(X), 128):
            idx = perm[i:i + 128]; loss = ((net(X[idx]) - Y[idx]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    net.eval()
    with torch.no_grad(): return {k: net(torch.tensor(v, device=dev)).cpu().numpy() for k, v in tests.items()}

In [ ]:
# 8. Part D (run). Assert the labels, train the three conditions, test on "real" images.
N_TRAIN, N_TEST, N_FLY, COND, SEEDS_D = 6000, 400, 40, ("fixed", "dr", "fit"), (0, 1, 2)

def partD():
    box = S.get("box", BOX); rng = np.random.default_rng(SEED)
    # (a) Assert the truth before use: the rendered mask has the label as its centre and the size of the scene's workpiece.
    pose = sample_pose(60, rng); m = coverage(pose, box[:2], 1.0); area = m.sum((1, 2))
    cx, cy = (m * GX).sum((1, 2)) / area, (m * GY).sum((1, 2)) / area
    assert np.abs(cx - pose[:, 0]).max() < 2e-3 and np.abs(cy - pose[:, 1]).max() < 2e-3, "mask centre is not the label"
    assert np.abs(area * (VIEW / NPX) ** 2 / (box[0] * box[1]) - 1).max() < 0.03, "mask area is not the workpiece area"
    print(f"label check on 60 poses: centre error {max(np.abs(cx - pose[:, 0]).max(), np.abs(cy - pose[:, 1]).max()) * 1000:.2f} mm, area error {np.abs(area * (VIEW / NPX) ** 2 / (box[0] * box[1]) - 1).max():.2%}")
    # (b) The data. Poses for the three tests differ from every training pose (different seeds).
    r = lambda k: np.random.default_rng(1000 * SEED + k)
    tp_sim, tp_real, tp_fly = sample_pose(N_TEST, r(1)), sample_pose(N_TEST, r(2)), sample_pose(N_FLY, r(3))
    tests = {"sim_clean": render_sim(tp_sim, sample_app("fixed", N_TEST, r(4)), r(5), box), "real": render_real(tp_real, r(6), box)}
    est = fit_look(render_real(tp_fly, r(7), box), tp_fly, box)
    print("flywheel estimate from 40 labelled real images: bg", np.round(est["bg"], 2), "obj", np.round(est["obj"], 2), "noise", round(est["noise"], 3))
    tr_pose = sample_pose(N_TRAIN, r(8)); Y = encode(tr_pose)
    X = {k: render_sim(tr_pose, sample_app(k, N_TRAIN, r(9), est), r(10), box) for k in COND}
    fig, ax = plt.subplots(3, 6, figsize=(9, 4.6))
    for i, (name, imgs) in enumerate((("fixed look", X["fixed"]), ("domain randomization", X["dr"]), ("'real' renderer", tests["real"]))):
        for j in range(6): ax[i, j].imshow(imgs[j].transpose(1, 2, 0)); ax[i, j].axis("off")
        ax[i, 0].set_title(name, fontsize=8, loc="left")
    fig.tight_layout(); fig.savefig(f"{W}/d_images.png", dpi=110); plt.close(fig)
    # (c) Train and test. Baseline: always answer the mean training pose.
    base = np.tile(Y.mean(0), (N_TEST, 1)); rows, per = [], {}
    for k in COND:
        for sd in SEEDS_D:
            t = time.time(); P = fit_cnn(X[k], Y, tests, sd)
            for name, tp in (("sim_clean", tp_sim), ("real", tp_real)):
                pos, ang = pose_err(P[name], tp); per[(k, sd, name)] = (pos, success(P[name], tp))
                rows.append(dict(train=k, seed=sd, test=name, pos_cm=pos.mean(), angle_deg=ang.mean(), n_ok=int(success(P[name], tp).sum())))
            print(f"trained {k} seed {sd} in {time.time() - t:.0f} s")
    raw = pd.DataFrame(rows); raw.round(3).to_csv(f"{W}/d_raw_rows.csv", index=False)
    out = []
    for k in COND:
        for name, tp in (("sim_clean", tp_sim), ("real", tp_real)):
            q = raw[(raw.train == k) & (raw.test == name)]
            out.append(dict(train=k, test=name, pos_cm=round(q.pos_cm.mean(), 2), angle_deg=round(q.angle_deg.mean(), 1), success=rate(q.n_ok.mean(), N_TEST),
                            success_by_seed="/".join(str(v) for v in q.n_ok)))
    for name, tp in (("sim_clean", tp_sim), ("real", tp_real)):
        pos, ang = pose_err(base, tp); out.append(dict(train="baseline: mean pose", test=name, pos_cm=round(pos.mean(), 2), angle_deg=round(ang.mean(), 1),
                                                       success=rate(success(base, tp).sum(), N_TEST)))
    show(pd.DataFrame(out), "d_results")
    # (d) Paired tests on the "real" images: the same 400 images, errors averaged over the three seeds.
    avg = lambda k: np.mean([per[(k, s, "real")][0] for s in SEEDS_D], 0)
    ok = lambda k: np.mean([per[(k, s, "real")][1] for s in SEEDS_D], 0) > .5
    cmp = []
    for a_, b_ in (("dr", "fixed"), ("fit", "dr")):
        d, lo, hi = paired_boot(avg(a_), avg(b_))
        cmp.append(dict(pair=f"{a_} minus {b_}", mean_pos_error_diff_cm=round(d, 2), ci95=f"[{lo:.2f}, {hi:.2f}]", mcnemar_p=round(mcnemar(ok(a_), ok(b_)), 4)))
    show(pd.DataFrame(cmp), "d_paired")
part("D synthetic data", partD, 1200)

## Part E: the three computers, and why the policy runs at the edge

Question: which computer does each job, and what does a network delay cost a control loop? `three-computers-robotics.txt` names the three computers: NVIDIA DGX for training, NVIDIA Omniverse and Cosmos on NVIDIA RTX PRO Servers for simulation, and NVIDIA Jetson AGX Thor for on-robot inference. In this notebook each job gets a stand-in.

| Computer | Job | Stand-in here |
|---|---|---|
| Training computer | Trains the policy | The T4: a small network learns from a teacher |
| Simulation computer | Tests the policy before it touches hardware | The twin from part C, with the loop in simulated time |
| Edge computer | Runs the policy next to the machine | One CPU thread, int8 weights, NumPy |

The task is a position move. The arm must turn to a random target angle and stay within 0.02 rad of it from 0.7 s to 1.2 s. Voltage is limited to 5 V. I set this spec after a NumPy check with PD laws, before I trained any network. A looser spec (0.03 rad, from 1.0 s) let both teachers pass on every machine, so the gap was invisible. The spec is a choice. The check shows that a design error can fail it. The machine has sensor noise. Each test uses 300 machines with hidden numbers within 5% of those in part C.

- **Policy**: a network with 2 inputs (angle error, speed), two tanh layers of 32 and one output. A teacher law (a PD law designed from a plant model) gives the target voltage. The network learns to copy the teacher on the T4. Two teachers are designed. One uses the datasheet numbers of the motor alone ("nominal"). One uses the numbers that the twin estimated in part C ("twin"). This is one turn of the flywheel.
- **Gap**: each policy runs in its own simulation, and on the hidden machines. The difference is the sim-to-real gap.
- **Edge**: the int8 model runs on one CPU thread. Its compute time is measured.
- **Cloud**: the same policy runs on the T4 behind a network. The loop gets a delay in simulated time: a queue of commands that are `delay` control periods old. No `sleep` call is used. The delay is the network round trip plus the measured T4 time, rounded to whole control periods of 10 ms.
- **Baseline**: always send 0 V.

In [ ]:
# 9. Part E (definitions). The loop in simulated time, graders, the int8 model, training on the T4, and timing.
T_EP, HOLD, TOL, HARD = 120, 50, 0.02, 2.0                             # ticks per episode (1.2 s), ticks that must stay in the band (0.7 s to 1.2 s), band (rad), hard stop (rad)
NOMINAL = (8.0, 1.0, 0.2)                                              # alpha, beta, gamma from the motor datasheet alone
BASE = (4.0, 2.5, 0.8)                                                 # the hidden family; used only to build the test machines

def rollout(policy, delay, par, target, seed):
    # Closed loop in simulated time. par = (alpha, beta, gamma), arrays over episodes. Returns the angle history (T_EP, n).
    a, b, g = par; rng = np.random.default_rng(seed); n = len(target)
    th, w, pipe, hist = np.zeros(n), np.zeros(n), [np.zeros(n) for _ in range(delay)], np.empty((T_EP, n))
    for t in range(T_EP):
        obs = np.column_stack([target - th - 0.002 * rng.standard_normal(n), w + 0.02 * rng.standard_normal(n)])
        pipe.append(policy(obs))                                       # the answer to the state seen at tick t
        u = np.clip(pipe.pop(0), -U_MAX, U_MAX)                        # the motor gets the answer that is `delay` ticks old
        w = w + DT * (a * u - b * w - g * np.tanh(w / EPS)); th = th + DT * w; hist[t] = th
    return hist
def settled(hist, target): return (np.abs(hist[-HOLD:] - target) < TOL).all(0)
def hard_hit(hist): return (np.abs(hist) > HARD).any(0)
_t = np.ones(3)
selfcheck("settled: right and wrong", settled(np.tile(_t, (T_EP, 1)), _t).all() and not settled(np.zeros((T_EP, 3)), _t).any())
selfcheck("hard_hit: right and wrong", hard_hit(np.full((T_EP, 3), 2.5)).all() and not hard_hit(np.ones((T_EP, 3))).any())
selfcheck("rollout: 0 V keeps the arm at rest", (rollout(lambda o: np.zeros(len(o)), 3, (np.ones(3), np.ones(3), np.zeros(3)), _t, 0)[-1] == 0).all())   # 0 V keeps the arm at rest

def machines(seed, n):
    r = np.random.default_rng(seed)
    return tuple(v * (1 + 0.05 * r.standard_normal(n)) for v in BASE), r.choice([-1, 1], n) * r.uniform(0.4, 1.2, n)

def gains(theta, wn=12.0, zeta=0.8):
    a, b, _ = theta; return wn ** 2 / a, (2 * zeta * wn - b) / a      # kp (V/rad), kd (V per rad/s) from pole placement
def pd_law(kp, kd): return lambda o: np.clip(kp * o[:, 0] - kd * o[:, 1], -U_MAX, U_MAX)

def q8(W): s = abs(W).max() / 127; return np.round(W / s).astype(np.int8), s
class Net:
    # The policy as NumPy code. quant=True: int8 weights, int8 activations, int32 sums. Only the scale and the tanh use floats.
    def __init__(self, layers, quant): self.q, self.L = quant, [(*(q8(W) if quant else (W, 1.0)), b) for W, b in layers]
    def __call__(self, obs):
        x = obs / np.array([1.5, 8.0])
        for i, (W, sw, b) in enumerate(self.L):
            if self.q:
                sx = np.maximum(abs(x).max(1, keepdims=True), 1e-9) / 127
                y = (np.round(x / sx).astype(np.int32) @ W.T.astype(np.int32)) * (sx * sw) + b
            else: y = x @ W.T + b
            x = np.tanh(y) if i < len(self.L) - 1 else y
        return U_MAX * np.tanh(x[:, 0])
_L = [(np.array([[1.0, 0.0], [0.0, 1.0]]), np.zeros(2)), (np.array([[1.0, 1.0]]), np.zeros(1))]
selfcheck("Net: int8 close to float, zero in zero out", abs(Net(_L, True)(np.array([[0.3, 0.8]])) - Net(_L, False)(np.array([[0.3, 0.8]]))).max() < 0.05 and Net(_L, False)(np.array([[0.0, 0.0]]))[0] == 0)

def train_policy(kp, kd, seed):
    # Copy a PD teacher into a small network on the GPU. Returns the NumPy layers, the torch module, and the final error in V^2.
    import torch, torch.nn as nn
    dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(seed); kp, kd = float(kp), float(kd)
    X = torch.rand(40000, 2, generator=torch.Generator().manual_seed(seed)).to(dev) * 2 - 1        # inputs scaled: error/1.5, speed/8
    U = torch.clamp(kp * 1.5 * X[:, 0] - kd * 8.0 * X[:, 1], -U_MAX, U_MAX)
    net = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1)).to(dev)
    opt = torch.optim.Adam(net.parameters(), 3e-3); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 3000)
    for i in range(3000):
        idx = torch.randint(0, len(X), (1024,), device=dev); loss = ((U_MAX * torch.tanh(net(X[idx])[:, 0]) - U[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    with torch.no_grad(): mse = float(((U_MAX * torch.tanh(net(X)[:, 0]) - U) ** 2).mean())
    lay = [(m.weight.detach().cpu().numpy().astype(np.float64), m.bias.detach().cpu().numpy().astype(np.float64)) for m in net if isinstance(m, nn.Linear)]
    return lay, net, mse

def gpu_ms(net):
    # One decision on the GPU with a batch of 1, including the copy of the input and the wait for the result.
    import torch
    dev = next(net.parameters()).device; x = torch.zeros(1, 2)
    with torch.no_grad():
        for _ in range(50): net(x.to(dev)).cpu()
        t = []
        for _ in range(300): t0 = time.perf_counter(); net(x.to(dev)).cpu(); t.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(t))

def edge_ms(net, n=20000):
    # One decision on one CPU thread. The process is pinned to CPU 0 for this loop only. Python overhead is included.
    old = os.sched_getaffinity(0) if hasattr(os, "sched_getaffinity") else None
    if old: os.sched_setaffinity(0, {min(old)})
    try:
        o, t = np.array([[0.2, 0.1]]), np.empty(n)
        for i in range(n): t0 = time.perf_counter_ns(); net(o); t[i] = time.perf_counter_ns() - t0
    finally:
        if old: os.sched_setaffinity(0, old)
    return np.percentile(t[500:], 50) / 1e6, np.percentile(t[500:], 99) / 1e6

In [ ]:
# 10. Part E (run). Train two policies, measure the gap, then compare the edge with the cloud.
def partE():
    theta = S["theta"]                                                  # fitted in part C from the sensor stream only
    nets, rows = {}, []
    for k, th in (("nominal", NOMINAL), ("twin", theta)):
        kp, kd = gains(th); t = time.time(); lay, net, mse = train_policy(kp, kd, SEED); nets[k] = (lay, net, kp, kd)
        rows.append(dict(teacher=k, alpha=round(float(th[0]), 2), beta=round(float(th[1]), 2), gamma=round(float(th[2]), 2), kp=round(kp, 1),
                         kd=round(kd, 2), train_s=round(time.time() - t, 1), mse_V2=round(mse, 4)))
    show(pd.DataFrame(rows), "e_training")
    N = 300; reps = {r: machines(r, N) for r in (11, 12)}
    def run(policy, delay, par=None, rep=11):
        p, tg = reps[rep]; p = par if par is not None else p
        h = rollout(policy, delay, p, tg, rep); return settled(h, tg), hard_hit(h)
    twin_par = tuple(np.full(N, float(v)) for v in theta); nom_par = tuple(np.full(N, float(v)) for v in NOMINAL)
    zero = lambda o: np.zeros(len(o))
    # (a) The gap: each policy in the simulation it was designed on, against the hidden machines. float model, no delay.
    gap = []
    for k, own in (("nominal", nom_par), ("twin", twin_par)):
        lay, _, kp, kd = nets[k]; net = Net(lay, False)
        for rep in reps:
            s_own, _ = run(net, 0, own, rep); s_real, h_real = run(net, 0, None, rep)
            gap.append(dict(policy=k, run=rep, own_simulation=rate(s_own.sum(), N), hidden_machines=rate(s_real.sum(), N), hard_stop_hits=int(h_real.sum()),
                            teacher_on_hidden=rate(run(pd_law(kp, kd), 0, None, rep)[0].sum(), N)))
    for rep in reps: gap.append(dict(policy="baseline: 0 V", run=rep, hidden_machines=rate(run(zero, 0, None, rep)[0].sum(), N)))
    show(pd.DataFrame(gap), "e_gap")
    # (b) Edge against cloud with the twin policy. Compute times are measured. The delay enters the loop in simulated ticks.
    lay, tnet, _, _ = nets["twin"]; f32, i8 = Net(lay, False), Net(lay, True)
    p50, p99 = edge_ms(i8); fp50, fp99 = edge_ms(f32, 5000); g_ms = gpu_ms(tnet)
    obs = np.random.default_rng(0).uniform(-1, 1, (5000, 2)) * [1.5, 8.0]
    print(f"int8 against float, 5,000 states: max |difference| = {abs(i8(obs) - f32(obs)).max():.3f} V (limit {U_MAX} V)")
    cfgs = [("edge", "float", 0.0, fp99, f32), ("edge", "int8", 0.0, p99, i8)] + [(f"cloud +{n} ms", "float", float(n), g_ms, f32) for n in (20, 50, 100)]
    res, ok_edge, out = [], {}, []
    for where, num, net_ms, comp_ms, net in cfgs:
        loop_ms = net_ms + comp_ms; ticks = int(round(loop_ms / (DT * 1000)))
        r = {rep: run(net, ticks, None, rep) for rep in reps}
        if (where, num) == ("edge", "int8"): ok_edge = {rep: r[rep][0] for rep in reps}
        out.append((where, num, p50 if num == "int8" else fp50, comp_ms, loop_ms, ticks, r))
    for where, num, c50, comp, loop, ticks, r in out:
        res.append(dict(where=where, numerics=num, compute_p50_ms=round(c50, 3) if where == "edge" else round(comp, 3), compute_p99_ms=round(comp, 3) if where == "edge" else None,
                        loop_delay_ms=round(loop, 1), ticks=ticks, run11=rate(r[11][0].sum(), N), run12=rate(r[12][0].sum(), N),
                        hard_stops_run11=int(r[11][1].sum()), p_vs_edge_int8=round(mcnemar(r[11][0], ok_edge[11]), 4)))
    show(pd.DataFrame(res), "e_edge_vs_cloud")
    # (c) Where does it break? Success against the loop delay, int8 policy.
    sweep = [(k, run(i8, k, None, 11)[0].sum()) for k in (0, 1, 2, 3, 4, 5, 7, 10, 15, 20)]
    show(pd.DataFrame([dict(delay_ms=int(k * DT * 1000), ticks=k, success=rate(v, N)) for k, v in sweep]), "e_delay_sweep")
    fig, ax = plt.subplots(figsize=(5, 3)); ax.plot([k * 10 for k, _ in sweep], [v / N for _, v in sweep], "o-"); ax.set_xlabel("loop delay (ms)")
    ax.set_ylabel("success"); ax.set_ylim(0, 1.02); fig.tight_layout(); fig.savefig(f"{W}/e_delay.png", dpi=110); plt.close(fig)
part("E three computers", partE, 480)

## Part F: physical AI against digital AI, and the model families

Question: how does physical AI differ from digital AI, and which NVIDIA model does which job? The exam asks for the jobs, so the cells below hold two tables from the sources and a matching quiz with an answer key. Every cell of the tables names its source file. A cell that says "not in the hub sources" marks a fact that no source states.

The quiz grader is deterministic. It gets one right key and one wrong key as asserts. The last line prints the score of a random guess, so you can compare it with your own.

In [ ]:
# 11. Part F. Two tables from the sources, and a matching quiz with a grader.
def partF():
    pa = pd.DataFrame([
        dict(topic="what goes in", physical_ai="images, video, language, speech, sensor data; GR00T: camera views, instruction, robot state",
             digital_ai="digital information", source="generative-physical-ai.txt, gr00t.txt"),
        dict(topic="what comes out", physical_ai="chunks of continuous robot actions (GR00T); here a motor command",
             digital_ai="not in the hub sources", source="gr00t.txt"),
        dict(topic="where it works", physical_ai="robots, vehicles, factories: the physical world",
             digital_ai="digital environments", source="three-computers-robotics.txt"),
        dict(topic="what a mistake costs", physical_ai="people hurt or equipment damaged, so test in simulation first",
             digital_ai="not in the hub sources", source="humanoid-robots.txt"),
        dict(topic="use cases", physical_ai="robots, robotaxis and vehicles, industrial AI; arms that adapt to part pose on a conveyor",
             digital_ai="not in the hub sources", source="generative-physical-ai.txt")])
    show(pa, "f_physical_vs_digital")
    fam = pd.DataFrame([
        dict(family="Cosmos Predict 2.5", kind="world foundation model", job="30-second videos from one frame, for world simulation", source="open-models-data-ai.txt"),
        dict(family="Cosmos Transfer 2.5", kind="world foundation model", job="photorealistic data from 3D scenes; scene augmentation", source="open-models-data-ai.txt"),
        dict(family="Cosmos Reason", kind="reasoning vision language model", job="multimodal understanding; a NIM microservice", source="open-models-data-ai.txt"),
        dict(family="Cosmos 3", kind="omni-model (Super, Nano, Edge)", job="reason, simulate worlds, generate data, specialize", source="cosmos.txt"),
        dict(family="Isaac GR00T", kind="open robot foundation models", job="camera views, instruction, state in; actions out", source="gr00t.txt"),
        dict(family="Isaac Sim", kind="simulation framework on Omniverse", job="train and test robots in simulation (not run here)", source="humanoid-robots.txt"),
        dict(family="Omniverse", kind="libraries and microservices", job="digital twins, robotics simulation (not run here)", source="ai-enterprise.txt")])
    show(fam, "f_model_families")
    QUIZ = [("It turns camera views, an instruction and the robot state into actions.", "Isaac GR00T"),
            ("It makes 30-second videos from one frame to simulate a world.", "Cosmos Predict"),
            ("It makes photorealistic data from 3D scenes.", "Cosmos Transfer"),
            ("It is a reasoning vision language model.", "Cosmos Reason"),
            ("The computer that trains the robot models.", "NVIDIA DGX"),
            ("The computer that runs the model on the robot.", "Jetson AGX Thor"),
            ("A robot simulation framework built on Omniverse.", "Isaac Sim"),
            ("It widens the generated data so that it covers the real range.", "domain randomization")]
    grade = lambda given, key: sum(g == k for g, k in zip(given, key)) / len(key)
    key = [a for _, a in QUIZ]
    assert grade(key, key) == 1.0 and grade(key[1:] + key[:1], key) == 0.0
    r = np.random.default_rng(SEED); rand = np.mean([grade(list(r.permutation(key)), key) for _ in range(2000)])
    print("Quiz. Match each clue to one answer. Answers:", sorted(key))
    for i, (c, _) in enumerate(QUIZ, 1): print(f"  {i}. {c}")
    print("Answer key:", {i: a for i, a in enumerate(key, 1)})
    print(f"score of a random matching over 2,000 tries: {rand:.3f} (expected 1/{len(key)} = {1 / len(key):.3f}). A blank answer scores 0.")
    assert abs(rand - 1 / len(key)) < 0.02
part("F physical AI", partF, 60)

## What to look for

- **A. Scene.** Check that the two source layers did not change, and that the override layer holds only the moved workpiece and the heavy payload. The variant changes the composed mass. The scene is the data format of Omniverse. Omniverse and Isaac Sim did not run.
- **B. Physics.** Read `b_checks`. Compare each error with its tolerance. A semi-implicit Euler step gives a known bias of about 1/n for n steps, which is 0.4% at 250 steps. Read the `source` column: `usd` means the engine loaded the scene file, `api` means the USD import failed and an equal scene was built in Python, and `kernel` means the Warp fallback ran. Read `b_speed`: one copy cannot fill a GPU, so the throughput per copy rises with the number of copies. A launch has a fixed cost. If Newton failed, the printed error above the table is the result.
- **C. Twin.** `c_estimate` shows how close the fit is to the hidden numbers. `c_follow` shows the twin following the wear when it refits on a window. In `c_alarm` read false alarms on the no-wear runs first, then detection and delay. With 30 runs, 0 false alarms still allows a true rate near 11% (Wilson upper bound). The raw baseline needs no twin but reads the command pattern as noise. `c_whatif` shows whether the twin predicted the real move time before the real machine ran.
- **D. Synthetic data.** Compare `sim_clean` and `real` for each training set. A model that scores well on its own renderer and badly on the other renderer shows the gap. Read `success_by_seed`: a difference smaller than the spread over seeds is noise. `d_paired` gives the paired interval on the same 400 images. The flywheel row used 40 labelled "real" images. The gap in this notebook is an appearance gap only. The camera geometry is the same in both renderers.
- **E. Three computers.** `e_gap` compares each policy in its own simulation and on the hidden machines. The nominal teacher uses the datasheet numbers and the twin teacher uses the fit from part C. `e_edge_vs_cloud` and `e_delay_sweep` show the delay at which the loop stops settling. The int8 model is checked against the float model. The edge time includes Python overhead and is not a Jetson time.
- **F. Physical AI.** Compare your quiz answers with the key.
- **Limits.** One machine type. A delay of 10 ms steps, so 20.4 ms rounds to 2 steps. Two repeats of 300 episodes. The "real" machine and the "real" images are simulators with hidden settings. A real robot adds effects that no model here contains. The Newton try uses an experimental path (USD 26.11 with Newton 1.8.0), and its result may be an error.

## Exam rules this proves

1. **physical.1**: Physical AI perceives, reasons and acts in the physical world. Digital AI works in digital environments. A physical system runs in a control loop with a fixed period, so a delay in the loop is a failure mode. Part E measures where the loop stops settling.
2. **physical.1**: The three-computer pattern is a training computer, a simulation computer and a computer on the robot (`three-computers-robotics.txt`). Part E gives each one a job: train on the T4, test in the twin, run at the edge.
3. **physical.1**: Edge inference keeps the decision next to the machine. The edge loop has no network delay. The cloud adds a network round trip to every decision. Compare the compute time with the 10 ms control period in `e_edge_vs_cloud`.
4. **physical.1**: Use cases named in the sources are robots, robotaxis and autonomous vehicles, and industrial AI. An arm that adjusts to the pose of a part on a conveyor is the case in parts D and E.
5. **physical.2**: A simulation computes how the machine moves. A digital twin is a simulation of one real machine, kept close to it with data. The twin in part C estimates the machine from sensors only, follows the wear, and answers a what-if that the real machine then confirms or not.
6. **physical.2**: OpenUSD holds the scene in layers, with references, variants and overrides. An override leaves the source layers unchanged. NVIDIA Omniverse reads this format. NVIDIA Isaac Sim is built on Omniverse. Neither ran here.
7. **physical.2**: Synthetic data has exact labels because the scene sets them. Domain randomization widens the generated data to cover the real range. The sim-to-real gap is the loss of accuracy on real data. Part D measures it and the effect of randomization.
8. **physical.2**: The sim-to-real flywheel logs real data, fits the simulator, narrows the randomization or the plant model, and retrains. Part D does one turn for images. Part E does one turn for the plant model, with the twin from part C.
9. **physical.3**: Cosmos is a platform of world foundation models and a reasoning model. Isaac GR00T is a family of open robot foundation models. The tables in part F give the job of each, with the source file.

| Exam term (objective) | Plain line | Shown by |
|---|---|---|
| Physical AI versus digital AI (physical.1) | Physical AI acts in the physical world. Digital AI works in digital environments. | Part F table, part E loop |
| Three-computer pattern (physical.1) | Train, simulate, run on the robot. | Part E table |
| Robotics and autonomous-machine use cases (physical.1) | Robots, vehicles, industrial AI. | Part F table, parts D and E |
| Edge inference (physical.1) | The model runs next to the machine. | Part E: edge against cloud |
| Simulation (physical.2) | A program that computes how a machine moves. | Parts B, C and E |
| Digital twins for robotics (physical.2) | A simulation of one real machine, kept close with data. | Part C |
| NVIDIA Omniverse (physical.2) | Libraries for digital twins and robotics simulation. Reads OpenUSD. Not run. | Part A (the format) |
| NVIDIA Isaac Sim (physical.2) | A robot simulation framework on Omniverse. Not run. | Part F table |
| Synthetic data (physical.2) | Data that a simulator makes, with exact labels. | Part D |
| Domain randomization (physical.2) | A new random look for every training sample. | Part D |
| The sim-to-real gap (physical.2) | The accuracy lost on real data. | Parts D and E |
| The sim-to-real flywheel (physical.2) | Real logs fit the simulator, then retrain. | Parts D and E |
| NVIDIA Physical AI model families and use cases (physical.3) | World models, a reasoning model, robot foundation models. | Part F table |
| NVIDIA Cosmos (physical.3) | World foundation models and a reasoning model. | Part F table and quiz |
| NVIDIA Isaac GR00T (physical.3) | Open robot foundation models: camera, instruction and state in, actions out. | Part F table and quiz |

In [ ]:
# 12. Status of all parts. No server runs in this notebook: Newton ran as a subprocess with a time limit. Print the GPU memory in use.
st = pd.DataFrame(STATUS)
show(st, "status")
print(f"self-checks passed: {sum(c['ok'] for c in SELF)}/{len(SELF)}", [c["check"] for c in SELF if not c["ok"]])
print(f"total wall time {(time.time() - T0) / 60:.1f} min; parts failed: {int((~st.ok).sum())}")
try:
    import torch; torch.cuda.empty_cache()
except Exception: pass
print("GPU memory in use (MiB):", sh(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], 30)[1].strip().replace("\n", " | "))
print("files in", W, ":", len(os.listdir(W)))